In [0]:
import requests
import json
import pandas as pd
from pyspark.sql.functions import col
import time

url = "https://jsearch.p.rapidapi.com/search-v2"

# Lista de países a consultar (máxima cobertura)
paises = [
    "us",  # Estados Unidos
    "ca",  # Canadá
    "gb",  # Reino Unido
    "mx",  # México
    "es",  # España
    "ar",  # Argentina
    "co",  # Colombia
    "cl",  # Chile
    "br",  # Brasil
    "pe",  # Perú
    "de",  # Alemania
    "fr",  # Francia
    "au",  # Australia
    "in",  # India
    "sg"   # Singapur
]
jobs_totales = []

print("Iniciando consulta a la API para múltiples países...")
print("=" * 80)

headers = {
	"x-rapidapi-key": "ea2dcb2dc4msh4b61325002ac817p1607e9jsn1f23fc582eae",
	"x-rapidapi-host": "jsearch.p.rapidapi.com",
	"Content-Type": "application/json"
}

for pais in paises:
    querystring = {
        "query": "Data Engineer",
        "num_pages": "10",  # Máximo de páginas por país
        "country": pais,
        "date_posted": "all"
    }
    
    try:
        print(f"\n📍 Consultando ofertas en: {pais.upper()}")
        response = requests.get(url, headers=headers, params=querystring)
        datos = response.json()

        # Extraer trabajos del país
        if datos.get('status') == 'OK' and 'data' in datos and 'jobs' in datos['data']:
            jobs_list = datos['data']['jobs']
            
            if jobs_list:
                # Agregar campo de país origen a cada registro
                for job in jobs_list:
                    job['source_country'] = pais
                
                jobs_totales.extend(jobs_list)
                print(f"   ✓ {len(jobs_list)} ofertas encontradas")
            else:
                print(f"   ⚠ No se encontraron ofertas")
        else:
            print(f"   ✗ Error en respuesta: {datos.get('status', 'Desconocido')}")
        
        # Pausa entre requests para no saturar la API
        time.sleep(1)
        
    except Exception as e:
        print(f"   ✗ Error al consultar {pais}: {str(e)}")
        continue

print("\n" + "=" * 80)
print(f"\n📊 RESUMEN FINAL")
print(f"   Total de ofertas obtenidas: {len(jobs_totales)}")
print(f"   Países consultados: {', '.join([p.upper() for p in paises])}")

# Crear DataFrame consolidado
if jobs_totales:
    df_pandas = pd.json_normalize(jobs_totales)
    df = spark.createDataFrame(df_pandas)
    
    # Mostrar resumen por país
    print(f"\n📈 Distribución por país:")
    df_country = df.groupBy('source_country').count().orderBy('count', ascending=False)
    display(df_country)
    
    # Mostrar muestra de datos
    columnas_principales = [
        'job_id', 'job_title', 'employer_name', 'source_country',
        'job_employment_type', 'job_country'
    ]
    columnas_existentes = [c for c in columnas_principales if c in df.columns]
    
    print(f"\n📋 Muestra de ofertas obtenidas:")
    display(df.select(*columnas_existentes).limit(20))
    
    print(f"\nTotal de columnas en el dataset: {len(df.columns)}")
else:
    print("\n⚠ No se obtuvieron ofertas de ningún país")

In [0]:
from datetime import datetime
from pyspark.sql.functions import col, regexp_replace, udf
from pyspark.sql.types import StringType
import json

# Definir la ruta del volumen
volume_path = "/Volumes/prueba_api/landing/archivos"

# Usar nombre fijo para permitir automatización
# El archivo siempre se sobrescribe en la misma ubicación
file_name = "data_engineer_jobs_latest.csv"
full_path = f"{volume_path}/{file_name}"

# Guardar timestamp y registros para logging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
total_registros = len(df_pandas)
print(f"📅 Timestamp de ejecución: {timestamp}")
print(f"📊 Registros en esta carga: {total_registros}")

# Función UDF para convertir listas/diccionarios a JSON string limpio
def to_json_string(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

to_json_udf = udf(to_json_string, StringType())

# Preparar el DataFrame desde pandas
df_spark = spark.createDataFrame(df_pandas)

# Procesar cada columna
for column_name in df_spark.columns:
    # Aplicar limpieza: convertir estructuras complejas a JSON y limpiar texto
    df_spark = df_spark.withColumn(
        column_name,
        regexp_replace(
            regexp_replace(
                to_json_udf(col(column_name)),
                "\\n", " "
            ),
            "\\r", ""
        )
    )

# Guardar el DataFrame como CSV
df_spark.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .option("escape", '"') \
    .csv(full_path)

print(f"✓ Archivo CSV guardado exitosamente en: {full_path}")
print(f"✓ Total de registros guardados: {df_spark.count()}")
print(f"✓ Columnas guardadas: {len(df_spark.columns)}")

# Obtener países incluidos (sin usar RDD, compatible con serverless)
paises_incluidos = [row.source_country for row in df_spark.select('source_country').distinct().collect()]
print(f"✓ Países incluidos: {', '.join(sorted(paises_incluidos))}")
print(f"✓ Todas las columnas limpias y formateadas correctamente")

In [0]:
# Listar archivos en el volumen
archivos = dbutils.fs.ls("/Volumes/prueba_api/landing/archivos")

print("Archivos en el volumen prueba_api.landing.archivos:")
print("=" * 80)
for archivo in archivos:
    print(f"Nombre: {archivo.name}")
    print(f"Ruta: {archivo.path}")
    print(f"Tamaño: {archivo.size / 1024:.2f} KB")
    print("-" * 80)